<a href="https://colab.research.google.com/github/saisathvik485/BIS-LAB-5M-/blob/main/cuckoo_search_using_network_router_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import random
import math
import networkx as nx
import numpy as np


def create_router_network():
    """Creates a simulated 10-router network with delay (ms), bandwidth (Mbps), and packet loss."""
    G = nx.Graph()

    # Format: (Router_A, Router_B, delay_ms, bandwidth_mbps, packet_loss_rate)
    links = [
        (0, 1, 5, 100, 0.01),
        (0, 2, 12, 50, 0.03),
        (0, 3, 8, 80, 0.02),
        (1, 4, 10, 100, 0.01),
        (1, 5, 15, 40, 0.05),
        (2, 4, 6, 90, 0.01),
        (2, 6, 14, 60, 0.04),
        (3, 5, 7, 100, 0.01),
        (3, 6, 9, 70, 0.02),
        (4, 7, 8, 100, 0.01),
        (5, 7, 11, 50, 0.03),
        (5, 8, 6, 90, 0.01),
        (6, 8, 10, 80, 0.02),
        (7, 9, 5, 100, 0.01),
        (8, 9, 7, 100, 0.01),
    ]

    for u, v, delay, bw, loss in links:
        G.add_edge(u, v, delay=delay, bandwidth=bw, loss=loss)

    return G


def calculate_qos_cost(G, path):
    """Evaluates path fitness (lower cost = better QoS)."""
    if not path or len(path) < 2:
        return float("inf")

    total_delay = 0.0
    min_bandwidth = float("inf")
    total_loss = 0.0

    for i in range(len(path) - 1):
        edge = G[path[i]][path[i + 1]]
        total_delay += edge["delay"]
        min_bandwidth = min(min_bandwidth, edge["bandwidth"])
        total_loss += edge["loss"]

    # Multi-objective QoS cost function
    w1, w2, w3, w4 = 0.4, 0.3, 0.2, 0.1
    cost = (
        w1 * total_delay
        + w2 * (1000.0 / min_bandwidth)
        + w3 * (total_loss * 100)
        + w4 * len(path)
    )
    return cost


def generate_random_path(G, source, target, max_steps=15):
    """Generates a valid loop-free path using randomized walk."""
    for _ in range(50):  # Retry limit to find valid path
        path = [source]
        visited = {source}
        curr = source

        while curr != target and len(path) < max_steps:
            neighbors = [n for n in G.neighbors(curr) if n not in visited]
            if not neighbors:
                break
            curr = random.choice(neighbors)
            path.append(curr)
            visited.add(curr)

        if curr == target:
            return path
    return nx.shortest_path(G, source, target)


def levy_flight_step(beta=1.5):
    """Generates a heavy-tailed Lévy flight step size."""
    sigma = (
        math.gamma(1 + beta)
        * np.sin(np.pi * beta / 2)
        / (
            math.gamma((1 + beta) / 2)
            * beta
            * (2 ** ((beta - 1) / 2))
        )
    ) ** (1 / beta)
    u = np.random.normal(0, sigma)
    v = np.random.normal(0, 1)
    return abs(u / (abs(v) ** (1 / beta)))


def mutate_path_via_levy(G, path, source, target):
    """Perturbs an existing route using Lévy flight step magnitude."""
    if len(path) <= 2:
        return generate_random_path(G, source, target)

    step = levy_flight_step()
    # Large Lévy jump rebuilds most of the path; small jump tweaks a sub-segment
    if step > 2.0:
        return generate_random_path(G, source, target)
    else:
        cut_idx = random.randint(1, len(path) - 2)
        prefix = path[:cut_idx]
        sub_path = generate_random_path(G, path[cut_idx], target)

        # Ensure loop-free combination
        combined = prefix + sub_path
        if len(combined) == len(set(combined)):
            return combined
        return generate_random_path(G, source, target)


def cuckoo_search_routing(
    G, source, target, num_nests=15, max_iter=50, pa=0.25
):
    """Runs Cuckoo Search to find the optimal QoS path between source and target routers."""
    nests = [generate_random_path(G, source, target) for _ in range(num_nests)]
    fitness = [calculate_qos_cost(G, p) for p in nests]

    best_idx = int(np.argmin(fitness))
    best_path = nests[best_idx]
    best_cost = fitness[best_idx]

    for _ in range(max_iter):
        # 1. Generate a new cuckoo solution via Lévy flight
        i = random.randint(0, num_nests - 1)
        new_path = mutate_path_via_levy(G, nests[i], source, target)
        new_cost = calculate_qos_cost(G, new_path)

        # 2. Choose a random nest j and replace if better
        j = random.randint(0, num_nests - 1)
        if new_cost < fitness[j]:
            nests[j] = new_path
            fitness[j] = new_cost

        # 3. Abandon a fraction (pa) of worst nests and build new ones
        num_abandon = max(1, int(pa * num_nests))
        worst_indices = np.argsort(fitness)[-num_abandon:]
        for idx in worst_indices:
            nests[idx] = generate_random_path(G, source, target)
            fitness[idx] = calculate_qos_cost(G, nests[idx])

        # 4. Update global best
        curr_best_idx = int(np.argmin(fitness))
        if fitness[curr_best_idx] < best_cost:
            best_cost = fitness[curr_best_idx]
            best_path = nests[curr_best_idx]

    return best_path, best_cost


if __name__ == "__main__":
    network = create_router_network()
    SOURCE_ROUTER = 0
    DEST_ROUTER = 9

    optimal_route, min_cost = cuckoo_search_routing(
        network, SOURCE_ROUTER, DEST_ROUTER, num_nests=15, max_iter=60
    )

    print(f"Source Router: {SOURCE_ROUTER} -> Destination Router: {DEST_ROUTER}")
    print(f"Optimal Router Path: {' -> '.join(map(str, optimal_route))}")
    print(f"Minimized QoS Cost:  {min_cost:.4f}")

Source Router: 0 -> Destination Router: 9
Optimal Router Path: 0 -> 1 -> 4 -> 7 -> 9
Minimized QoS Cost:  15.5000
